# Agentic Market Research Project

## Step-1: Installing Dependencies

In [ ]:
!pip install -q "google-genai" "duckduckgo-search" "pandas==2.2.3" "google-auth==2.49.0" "reportlab"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 118.6 MB/s eta 0:00:00


## Step 2: Setting Up Gemini API Key

In [ ]:
import os
from google import genai

# Set API key directly or via Colab Secrets
os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6I50Y9Y8qdS_WJfkjsBCSQKBYGpTmibAD6WNvXgWf8V6A"

# Initialize Google GenAI client
client = genai.Client()

## Step 3: Defining Tools & Multi-Agent Architecture

In [ ]:
import json
import pandas as pd
from duckduckgo_search import DDGS

# ---------------------------------------------------------
# Tool: Free DuckDuckGo Web Search
# ---------------------------------------------------------
def web_search(query: str, max_results: int = 5) -> str:
    """Searches the live web for market data using DuckDuckGo."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
            return json.dumps(results, indent=2)
    except Exception as e:
        return f"Search failed: {str(e)}"

# ---------------------------------------------------------
# Agent Pipeline Class
# ---------------------------------------------------------
class MarketResearchWorkflow:
    def __init__(self, model_name: str = "gemini-2.5-flash"):
        self.model = model_name

    def research_agent(self, company_or_industry: str) -> str:
        """Agent 1: Gathers live web market data."""
        print("🔍 Agent 1 (Researcher): Gathering live market data...")

        search_query = f"{company_or_industry} market size competitors growth news"
        raw_data = web_search(search_query, max_results=6)

        prompt = f"""
        You are a Senior Market Research Analyst. Analyze the raw search data below for target: '{company_or_industry}'.
        Extract key factual details including: market size, major competitors, key trends, and recent risks or opportunities.

        Raw Data:
        {raw_data}
        """

        response = client.models.generate_content(
            model=self.model,
            contents=prompt,
        )
        return response.text

    def analyst_agent(self, research_summary: str) -> str:
        """Agent 2: Conducts SWOT analysis and financial synthesis."""
        print("📊 Agent 2 (Analyst): Performing Strategic Analysis (SWOT)...")

        prompt = f"""
        You are a Chief Financial & Strategy Analyst. Based on the market research findings provided, perform a comprehensive analysis.

        Structure your response with:
        1. Executive Summary
        2. SWOT Analysis (Strengths, Weaknesses, Opportunities, Threats)
        3. Competitive Landscape Overview

        Research Data:
        {research_summary}
        """

        response = client.models.generate_content(
            model=self.model,
            contents=prompt,
        )
        return response.text

    def strategist_agent(self, strategic_analysis: str) -> str:
        """Agent 3: Formulates actionable strategy recommendations."""
        print("💡 Agent 3 (Strategist): Generating Actionable Recommendations...")

        prompt = f"""
        You are a Management Consultant. Based on the SWOT and analysis below, provide 3 to 5 concrete, prioritized strategic recommendations for growth and risk mitigation.

        Analysis:
        {strategic_analysis}
        """

        response = client.models.generate_content(
            model=self.model,
            contents=prompt,
        )
        return response.text

    def run_pipeline(self, target: str) -> dict:
        """Executes the sequential multi-agent workflow."""
        print(f"🚀 Starting Agentic Market Research for: '{target}'\n" + "="*50)

        # Step 1: Research
        research_data = self.research_agent(target)

        # Step 2: Analysis
        analysis_data = self.analyst_agent(research_data)

        # Step 3: Strategy
        strategy_data = self.strategist_agent(analysis_data)

        print("✅ Workflow Complete!\n" + "="*50)

        return {
            "target": target,
            "research": research_data,
            "analysis": analysis_data,
            "strategy": strategy_data
        }

## Step 4: Execute & Render Results

In [ ]:
import re
from google.colab import files
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer

# ---------------------------------------------------------
# Custom Canvas for Page Numbers & Headers
# ---------------------------------------------------------
class NumberedCanvas(canvas.Canvas):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._saved_page_states = []

    def showPage(self):
        self._saved_page_states.append(dict(self.__dict__))
        self._startPage()

    def save(self):
        num_pages = len(self._saved_page_states)
        for state in self._saved_page_states:
            self.__dict__.update(state)
            self.draw_page_number(num_pages)
            super().showPage()
        super().save()

    def draw_page_number(self, page_count):
        self.saveState()
        self.setFont("Helvetica", 8)
        self.setFillColor(colors.HexColor("#64748B"))

        # Header (pages > 1)
        if self._pageNumber > 1:
            self.drawString(54, 11 * inch - 36, "Agentic Market Research Report")
            self.setStrokeColor(colors.HexColor("#CBD5E1"))
            self.setLineWidth(0.5)
            self.line(54, 11 * inch - 42, 8.5 * inch - 54, 11 * inch - 42)

        # Footer
        footer_text = f"Page {self._pageNumber} of {page_count}"
        self.drawRightString(8.5 * inch - 54, 36, footer_text)
        self.drawString(54, 36, "Confidential — Generated via Gemini Multi-Agent System")
        self.setStrokeColor(colors.HexColor("#CBD5E1"))
        self.setLineWidth(0.5)
        self.line(54, 48, 8.5 * inch - 54, 48)

        self.restoreState()

# ---------------------------------------------------------
# Markdown-to-ReportLab Helper Function
# ---------------------------------------------------------
def md_to_reportlab(text: str, normal_style, heading2_style, heading3_style):
    """Converts basic markdown formatting into ReportLab Flowable objects."""
    flowables = []
    lines = text.strip().split("\n")

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Convert Markdown bold and italic to HTML tags for ReportLab
        line = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', line)
        line = re.sub(r'\*(.*?)\*', r'<i>\1</i>', line)

        # Headings
        if line.startswith("### "):
            flowables.append(Paragraph(line[4:], heading3_style))
            flowables.append(Spacer(1, 4))
        elif line.startswith("## ") or line.startswith("# "):
            clean_title = re.sub(r'^[#\s]+', '', line)
            flowables.append(Paragraph(clean_title, heading2_style))
            flowables.append(Spacer(1, 6))
        # Bullet points
        elif line.startswith("- ") or line.startswith("* "):
            bullet_text = f"• {line[2:]}"
            flowables.append(Paragraph(bullet_text, normal_style))
            flowables.append(Spacer(1, 4))
        # Numbered lists
        elif re.match(r'^\d+\.\s', line):
            flowables.append(Paragraph(line, normal_style))
            flowables.append(Spacer(1, 4))
        # Regular paragraphs
        else:
            flowables.append(Paragraph(line, normal_style))
            flowables.append(Spacer(1, 6))

    return flowables

# ---------------------------------------------------------
# PDF Generator Function
# ---------------------------------------------------------
def generate_pdf_report(results: dict, output_filename: str = "Market_Research_Report.pdf"):
    print("📄 Building PDF document...")

    doc = SimpleDocTemplate(
        output_filename,
        pagesize=letter,
        leftMargin=54,
        rightMargin=54,
        topMargin=54,
        bottomMargin=54
    )

    styles = getSampleStyleSheet()

    # Custom Palette & Typography
    title_style = ParagraphStyle(
        'DocTitle',
        parent=styles['Normal'],
        fontName='Helvetica-Bold',
        fontSize=22,
        leading=26,
        textColor=colors.HexColor('#1E293B'),
        spaceAfter=12
    )

    subtitle_style = ParagraphStyle(
        'DocSubtitle',
        parent=styles['Normal'],
        fontName='Helvetica',
        fontSize=12,
        leading=16,
        textColor=colors.HexColor('#64748B'),
        spaceAfter=20
    )

    h1_style = ParagraphStyle(
        'Heading1_Custom',
        parent=styles['Normal'],
        fontName='Helvetica-Bold',
        fontSize=15,
        leading=18,
        textColor=colors.HexColor('#0F172A'),
        spaceBefore=14,
        spaceAfter=8,
        keepWithNext=True
    )

    h2_style = ParagraphStyle(
        'Heading2_Custom',
        parent=styles['Normal'],
        fontName='Helvetica-Bold',
        fontSize=12,
        leading=15,
        textColor=colors.HexColor('#2563EB'),
        spaceBefore=10,
        spaceAfter=6,
        keepWithNext=True
    )

    body_style = ParagraphStyle(
        'Body_Custom',
        parent=styles['Normal'],
        fontName='Helvetica',
        fontSize=10,
        leading=14,
        textColor=colors.HexColor('#334155'),
        spaceAfter=4
    )

    story = []

    # Title Block
    story.append(Paragraph("Market Research & Strategic Analysis", title_style))
    story.append(Paragraph(f"<b>Target Subject:</b> {results['target']}", subtitle_style))
    story.append(Spacer(1, 10))

    # Section 1: Executive Findings
    story.append(Paragraph("1. Executive Research & Findings", h1_style))
    story.extend(md_to_reportlab(results['research'], body_style, h2_style, body_style))
    story.append(Spacer(1, 12))

    # Section 2: SWOT Analysis
    story.append(Paragraph("2. SWOT & Strategic Analysis", h1_style))
    story.extend(md_to_reportlab(results['analysis'], body_style, h2_style, body_style))
    story.append(Spacer(1, 12))

    # Section 3: Recommendations
    story.append(Paragraph("3. Strategic Recommendations", h1_style))
    story.extend(md_to_reportlab(results['strategy'], body_style, h2_style, body_style))

    # Build PDF
    doc.build(story, canvasmaker=NumberedCanvas)
    print(f"✅ PDF successfully generated: {output_filename}")


# ---------------------------------------------------------
# Execution Flow
# ---------------------------------------------------------

# 1. Initialize & Run Workflow
workflow = MarketResearchWorkflow(model_name="gemini-3.6-flash")
TARGET_SUBJECT = "AI_Powered_Customer_Support"
results = workflow.run_pipeline(TARGET_SUBJECT)

# 2. Generate PDF File
pdf_filename = f"{TARGET_SUBJECT.replace(' ', '_')}_Report.pdf"
generate_pdf_report(results, output_filename=pdf_filename)

# 3. Auto-Download in Colab
print("📥 Triggering browser download...")
files.download(pdf_filename)

🚀 Starting Agentic Market Research for: 'AI_Powered_Customer_Support'
🔍 Agent 1 (Researcher): Gathering live market data...


/tmp/ipykernel_4002/2833250241.py:11: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use tim

📊 Agent 2 (Analyst): Performing Strategic Analysis (SWOT)...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

💡 Agent 3 (Strategist): Generating Actionable Recommendations...


...

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag